# Merge Signals → Unified Dataset

Merges all signal CSVs with pressure data into a single parquet file.
Run this after the pipeline finishes to verify `team_polarisation` is no longer NaN.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Find project root
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Merge all signals into unified dataset
from src.merge_outputs import merge_all

unified = merge_all(
    signals_dir=str(PROJECT_ROOT / "outputs" / "signals"),
    pressure_dir=str(PROJECT_ROOT / "outputs" / "pressure_exposure"),
    output_path=str(PROJECT_ROOT / "outputs" / "unified_fatigue_dataset.parquet"),
)
print(f"\n✅ Done: {len(unified)} rows × {len(unified.columns)} columns")

In [ ]:
# Check: is team_polarisation still all NaN?
nan_count = unified.team_polarisation.isna().sum()
total = len(unified)
print(f"team_polarisation NaN: {nan_count} / {total} rows ({100*nan_count/total:.1f}%)")
print(f"team_polarisation values: {sorted(unified.team_polarisation.dropna().unique())[:10]}")

if nan_count < total:
    print("✅ Fix working — polarisation has real values!")
else:
    print("❌ Still all NaN — check the signal CSVs exist in outputs/signals/team_polarisation/")

In [ ]:
# Quick peek at the data
cols = [c for c in ['game_id','block_id','player_id','team_id_opta',
                     'team_polarisation','team_centroid_distance',
                     'reorientation_rate','shift_latency'] if c in unified.columns]
unified[cols].head(10)